
# MobileNetV2 Transfer Learning — PyTorch & TensorFlow/Keras
Created on 2025-12-09 09:20:32.  
This notebook includes **both** frameworks — pick one path and run the cells below it.

> Contents:
> 1. Setup & Utilities  
> 2. **PyTorch**: Training on `ImageFolder`, Evaluation, Inference, Export (TorchScript)  
> 3. **TensorFlow/Keras**: Training on `image_dataset_from_directory`, Fine-tuning, Inference, Export (SavedModel/TFLite)


## 1) Setup & Utilities

In [ ]:

# !pip -q install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip -q install tensorflow==2.16.1  # adjust for your environment/GPU
# !pip -q install pillow matplotlib

import os, time, json, math, random, shutil, sys
from pathlib import Path

print("Python:", sys.version)
print("CUDA visible devices:", os.environ.get("CUDA_VISIBLE_DEVICES"))



### Expected dataset layout

For **both** frameworks we use a class-per-folder layout.

```
data/
  train/
    classA/ img1.jpg, img2.jpg, ...
    classB/ ...
  val/
    classA/ ...
    classB/ ...
```
Change the `DATA_DIR` below if your path differs.


In [ ]:

# Configure paths
DATA_DIR = ""   # change me if needed
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10

# Create dirs if you want to quickly sanity-check with toy data later
Path(DATA_DIR).mkdir(exist_ok=True)
print("Using DATA_DIR =", os.path.abspath(DATA_DIR))



---
## 2) PyTorch — MobileNetV2
This section uses `torchvision.datasets.ImageFolder` and pretrained MobileNetV2.


In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| device:", device)

def get_loaders(data_dir, batch_size=32, img_size=224, workers=2):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    train_tfms = transforms.Compose([
        transforms.RandomResizedCrop(img_size),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2,0.2,0.2,0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    val_tfms = transforms.Compose([
        transforms.Resize(int(img_size*1.15)),
        transforms.CenterCrop(img_size),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    train_ds = datasets.ImageFolder(os.path.join(data_dir, "train"), train_tfms)
    val_ds   = datasets.ImageFolder(os.path.join(data_dir, "val"),   val_tfms)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=workers, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=workers, pin_memory=True)
    return train_dl, val_dl, train_ds.classes

train_dl, val_dl, class_names = get_loaders(DATA_DIR, BATCH_SIZE, IMG_SIZE)
num_classes = len(class_names)
print("Classes:", class_names)


In [ ]:

# Load pretrained MobileNetV2 and replace the classifier head
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
for p in model.features.parameters():
    p.requires_grad = True  # set False for quick freezing

in_feat = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_feat, num_classes)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def train_one_epoch(model, dl, criterion, optimizer, device):
    model.train()
    total_loss, total_correct = 0.0, 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * xb.size(0)
        total_correct += (out.argmax(1) == yb).sum().item()
    n = len(dl.dataset)
    return total_loss/n, total_correct/n

@torch.no_grad()
def eval_one_epoch(model, dl, criterion, device):
    model.eval()
    total_loss, total_correct = 0.0, 0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = criterion(out, yb)
        total_loss += loss.item() * xb.size(0)
        total_correct += (out.argmax(1) == yb).sum().item()
    n = len(dl.dataset)
    return total_loss/n, total_correct/n


In [ ]:

best_acc = 0.0
SAVE_PATH = "mobilenet_v2_best.pth"

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_dl, criterion, optimizer, device)
    va_loss, va_acc = eval_one_epoch(model, val_dl, criterion, device)
    scheduler.step()
    print(f"Epoch {epoch:02d} | Train loss {tr_loss:.4f} acc {tr_acc:.4f} | Val loss {va_loss:.4f} acc {va_acc:.4f}  "
          f"({time.time()-t0:.1f}s)")
    if va_acc > best_acc:
        best_acc = va_acc
        torch.save({"model": model.state_dict(), "classes": class_names}, SAVE_PATH)
        print(f"  ✔ New best val acc {best_acc:.4f}. Saved -> {SAVE_PATH}")


In [ ]:

# Single-image inference (PyTorch)
from PIL import Image
from torchvision import transforms

CKPT = "mobilenet_v2_best.pth"
IMG  = "test.jpg"   # change to your image path

state = torch.load(CKPT, map_location="cpu")
classes = state["classes"]

infer_model = models.mobilenet_v2(weights=None)
in_feat = infer_model.classifier[1].in_features
infer_model.classifier[1] = nn.Linear(in_feat, len(classes))
infer_model.load_state_dict(state["model"])
infer_model.eval().to(device)

tfm = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

if os.path.exists(IMG):
    img = Image.open(IMG).convert("RGB")
    x = tfm(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = infer_model(x)
        probs = logits.softmax(dim=1).squeeze(0).cpu().numpy()
    idx = int(probs.argmax())
    print(f"Pred: {classes[idx]} (conf={probs[idx]:.3f})")
else:
    print(f"Image not found: {IMG}")


In [ ]:

# Export to TorchScript
infer_model.eval().cpu()
dummy = torch.randn(1,3,224,224)
traced = torch.jit.trace(infer_model, dummy)
traced.save("mobilenet_v2_traced.pt")
print("Saved TorchScript to mobilenet_v2_traced.pt")



---
## 3) TensorFlow/Keras — MobileNetV2
This section uses `tf.keras.applications.MobileNetV2` with preprocessing, optional fine-tuning, and exports.


In [ ]:

import tensorflow as tf
from tensorflow.keras import layers, models

print("TensorFlow:", tf.__version__)

IMG_SIZE = int(IMG_SIZE)  # reuse same global size

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(DATA_DIR, "train"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    os.path.join(DATA_DIR, "val"),
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

class_names_keras = train_ds.class_names
num_classes_keras = len(class_names_keras)
print("Classes:", class_names_keras)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)


In [ ]:

def build_keras_mobilenetv2(num_classes, img_size=224, freeze_backbone=True):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(img_size, img_size, 3),
        include_top=False,
        weights="imagenet",
        pooling="avg")
    base.trainable = not freeze_backbone

    inputs = layers.Input(shape=(img_size, img_size, 3))
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base(x, training=False)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(3e-4),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model, base

keras_model, base = build_keras_mobilenetv2(num_classes_keras, IMG_SIZE, freeze_backbone=True)
keras_model.summary()


In [ ]:

history = keras_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)
keras_model.save("mobilenet_v2_keras.h5")
with open("classes_keras.txt","w",encoding="utf-8") as f:
    f.write("\n".join(class_names_keras))
print("Saved model to mobilenet_v2_keras.h5")


In [ ]:

# Optional fine-tuning: unfreeze last N layers
FINE_TUNE = True
UNFREEZE_LAYERS = 30  # last 30 layers

if FINE_TUNE:
    base.trainable = True
    for layer in base.layers[:-UNFREEZE_LAYERS]:
        layer.trainable = False

    keras_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                        loss="sparse_categorical_crossentropy",
                        metrics=["accuracy"])
    history_ft = keras_model.fit(train_ds, validation_data=val_ds, epochs=max(3, EPOCHS//2))
    keras_model.save("mobilenet_v2_keras_finetuned.h5")
    print("Saved fine-tuned model to mobilenet_v2_keras_finetuned.h5")


In [ ]:

# Single-image inference (Keras)
from PIL import Image
import numpy as np

MODEL_PATH = "mobilenet_v2_keras.h5"  # or "mobilenet_v2_keras_finetuned.h5"
IMG = "test.jpg"  # change to your image path

if os.path.exists(MODEL_PATH):
    model = tf.keras.models.load_model(MODEL_PATH)
    classes = [c.strip() for c in open("classes_keras.txt","r",encoding="utf-8").read().splitlines()]
    img = Image.open(IMG).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    x = np.array(img, dtype=np.float32)[None, ...]
    x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
    probs = model.predict(x, verbose=0)[0]
    idx = int(np.argmax(probs))
    print(f"Pred: {classes[idx]} (conf={float(probs[idx]):.3f})")
else:
    print(f"Model not found: {MODEL_PATH}")


In [ ]:

# Export to SavedModel and TFLite
EXPORT_DIR = "saved_mobilenet_v2"
tf.saved_model.save(keras_model, EXPORT_DIR)
converter = tf.lite.TFLiteConverter.from_saved_model(EXPORT_DIR)
tflite_model = converter.convert()
open("mobilenet_v2.tflite","wb").write(tflite_model)
print("Saved SavedModel to", EXPORT_DIR, "and TFLite to mobilenet_v2.tflite")



### Tips
- Start with the backbone frozen for quicker convergence; then fine-tune top layers with a smaller learning rate.
- Keep input size at **224×224** to match MobileNetV2 defaults.
- If your dataset is small, increase regularization (e.g., dropout) and augmentation.
- Exports:
  - **PyTorch** → `mobilenet_v2_traced.pt` (TorchScript)
  - **Keras** → SavedModel in `saved_mobilenet_v2/` and `mobilenet_v2.tflite` for mobile/edge
